# Lab Activity 5: Building and Evaluating Homology Models of Myoglobin

Run the cells in order. Follow the instructions on the Lab Activity 5 page for each step, answer the questions there, and paste your Modeller license key only in the first cell. Do not share this notebook with your key in it.

## 2. Activity 1: Set up Modeller in Google Colab

In [ ]:
import os
KEY = "YOUR_KEY_HERE"          # your Modeller license key, from the registration email
!wget -q https://salilab.org/modeller/10.8/modeller-10.8.tar.gz
!tar -zxf modeller-10.8.tar.gz
os.chdir("/content/modeller-10.8")
with open("modeller_config", "w") as f:
    f.write("2\n/content/compiled/MODELLER\n" + KEY + "\n")
!./Install < modeller_config > install.log
!ln -sf /content/compiled/MODELLER/bin/mod10.8 /usr/bin/mod10.8
os.chdir("/content")
!pip -q install biopython
!mod10.8 2>&1 | head -1

In [ ]:
from Bio.PDB import PDBList, PDBParser, PDBIO, Select
class ChainA(Select):
    def accept_chain(self, chain): return chain.id == "A"
pdbl = PDBList()
for pid in ["1A6M", "1LHS", "2NRL", "1WLA"]:
    pdbl.retrieve_pdb_file(pid, pdir=".", file_format="pdb", overwrite=True)
    s = PDBParser(QUIET=True).get_structure(pid, "pdb%s.ent" % pid.lower())
    io = PDBIO(); io.set_structure(s); io.save(pid + ".pdb", ChainA())
    n = sum(1 for r in s[0]["A"] if r.id[0] == " ")
    print(pid, "chain A,", n, "residues")

## 3. Activity 2: Get the target sequence and align it to the template

In [ ]:
import urllib.request
fasta = urllib.request.urlopen("https://rest.uniprot.org/uniprotkb/P68082.fasta").read().decode()
seq = "".join(fasta.split("\n")[1:]).strip()
print(len(seq), seq[:10])
seq = seq[1:]                       # drop the initial methionine
print(len(seq), seq[:10])
with open("horse_mb.ali", "w") as f:
    f.write(">P1;horse_mb\nsequence:horse_mb:::::::0.00: 0.00\n" + seq + "*\n")

In [ ]:
%%writefile align_1A6M.py
from modeller import *
env = Environ()
aln = Alignment(env)
mdl = Model(env, file="1A6M", model_segment=("FIRST:A", "LAST:A"))
aln.append_model(mdl, align_codes="1A6MA", atom_files="1A6M.pdb")
aln.append(file="horse_mb.ali", align_codes="horse_mb")
aln.align2d(max_gap_length=50)
aln.write(file="horse-1A6M.ali", alignment_format="PIR")
aln.write(file="horse-1A6M.pap", alignment_format="PAP")

In [ ]:
!mod10.8 align_1A6M.py
!cat horse-1A6M.pap

## 4. Activity 3: Build five models and rank them

In [ ]:
%%writefile build_1A6M.py
from modeller import *
from modeller.automodel import *
env = Environ()
a = AutoModel(env, alnfile="horse-1A6M.ali", knowns="1A6MA", sequence="horse_mb",
              assess_methods=(assess.DOPE, assess.GA341))
a.starting_model = 1
a.ending_model = 5
a.make()
ok = [x for x in a.outputs if x["failure"] is None]
ok.sort(key=lambda x: x["DOPE score"])
print("model                    molpdf      DOPE    GA341")
for x in ok:
    print("%-22s %9.1f %9.1f %8.3f" % (x["name"], x["molpdf"], x["DOPE score"], x["GA341 score"][0]))

In [ ]:
!mod10.8 build_1A6M.py
!tail -8 build_1A6M.log

## 5. Activity 4: Evaluate the best model

In [ ]:
%%writefile dope_1A6M.py
from modeller import *
from modeller.scripts import complete_pdb
env = Environ()
env.libs.topology.read(file="$(LIB)/top_heav.lib")
env.libs.parameters.read(file="$(LIB)/par.lib")
for name, fn in [("model", "horse_mb.B99990003.pdb"), ("template", "1A6M.pdb")]:
    mdl = complete_pdb(env, fn)
    s = Selection(mdl.chains[0])
    s.assess_dope(output="ENERGY_PROFILE NO_REPORT", file=name + ".profile",
                  normalize_profile=True, smoothing_window=15)
    print(name, "normalized DOPE z-score: %.2f" % mdl.assess_normalized_dope())

In [ ]:
!mod10.8 dope_1A6M.py
!grep "z-score" dope_1A6M.log

In [ ]:
import matplotlib.pyplot as plt
def read_profile(fn):
    vals = [float(l.split()[-1]) for l in open(fn) if not l.startswith("#") and len(l.split()) > 1]
    return vals
m, t = read_profile("model.profile"), read_profile("template.profile")
plt.figure(figsize=(9, 4))
plt.plot(range(1, len(m) + 1), m, label="model (horse)", color="#0D776B")
plt.plot(range(1, len(t) + 1), t, label="template (1A6M)", color="#9EA3A8")
plt.xlabel("residue"); plt.ylabel("DOPE per residue (smoothed)"); plt.legend(); plt.show()

Run this in the VMD Tk Console, not in Colab:

```tcl
set ref [atomselect 0 "name CA"]
set mob [atomselect 1 "name CA"]
set all [atomselect 1 "all"]
puts "atoms: [$ref num] and [$mob num]"
set M [measure fit $mob $ref]
$all move $M
puts "RMSD after fit: [measure rmsd $mob $ref]"
```

## 6. Activity 5: Repeat with the sea turtle and tuna templates

Move the whale models into their own folder before the next run, so they are not overwritten. Then copy the cells of Activities 2 to 4 below, replace `1A6M` with `1LHS` (and the align code `1A6MA` with `1LHSA`), and run them. Afterwards move those models to a folder `turtle` and repeat with `2NRL`.

In [ ]:
!mkdir -p whale && mv horse_mb.B9999* whale/
!ls whale

In [ ]:
# paste and edit the Activity 2 to 4 cells here for 1LHS

In [ ]:
!mkdir -p turtle && mv horse_mb.B9999* turtle/
!ls turtle

In [ ]:
# paste and edit the Activity 2 to 4 cells here for 2NRL

## 8. Activity 6: Check an AI-written evaluation script

Paste the script your AI assistant wrote (prompt on the lab page, Section 8.1) into the cell below, read it, run it, and compare the RMSD with your VMD value.

In [ ]:
# AI-written script goes here